## Step 1: Install and Import Required Libraries

This cell installs (if necessary) and imports all the required Python packages for BERT model training and evaluation using Hugging Face and common machine learning tools.  
- **Hugging Face Transformers** for BERT & model management
- **Datasets** for data handling
- **Evaluate** for metrics
- **PyTorch** for deep learning
- **NumPy/Pandas/Scikit-learn** for data science basics


In [12]:
# Run this cell first!
!pip install --upgrade "accelerate>=1.1.0" transformers torch evaluate

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
import evaluate
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import torch
import numpy as np

In [13]:
# Adjust path to your CSV as needed
df = pd.read_csv('../../src/merged_and_cleaned.csv')
print(df.head())
print(df['label'].value_counts())

# Small, balanced sample for CPU training
SAMPLE_SIZE = 5000  # Increase if you have more RAM!
fake = df[df['label'] == 1].sample(SAMPLE_SIZE // 2, random_state=42)
real = df[df['label'] == 0].sample(SAMPLE_SIZE // 2, random_state=42)
df_small = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_small['label'].value_counts())

           text  label
0  fake_webpage      1
1  fake_webpage      1
2  fake_webpage      1
3  fake_webpage      1
4  fake_webpage      1
label
0    30865
1    27246
Name: count, dtype: int64
label
1    2500
0    2500
Name: count, dtype: int64


In [14]:
train_df, val_df = train_test_split(
    df_small, test_size=0.2, stratify=df_small['label'], random_state=42
)
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}")

Train size: 4000, Validation size: 1000


In [15]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(example):
    return tokenizer(
        example["text"], padding="max_length", truncation=True, max_length=128
    )

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

train_ds = train_ds.rename_column('label', 'labels')
val_ds = val_ds.rename_column('label', 'labels')
train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [16]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,               # Do one epoch at a time for incremental training!
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",            # Save after each epoch
    save_total_limit=2,               # Keep last 2 checkpoints only
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [17]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,               # Do one epoch at a time for incremental training!
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",            # Save after each epoch
    save_total_limit=2,               # Keep last 2 checkpoints only
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [18]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,               # Do one epoch at a time for incremental training!
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",            # Save after each epoch
    save_total_limit=2,               # Keep last 2 checkpoints only
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [19]:
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,               # Do one epoch at a time for incremental training!
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir='./logs',
    save_strategy="epoch",            # Save after each epoch
    save_total_limit=2,               # Keep last 2 checkpoints only
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

In [20]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    result = {}
    result.update(accuracy.compute(predictions=predictions, references=labels))
    result.update(precision.compute(predictions=predictions, references=labels, average='binary'))
    result.update(recall.compute(predictions=predictions, references=labels, average='binary'))
    result.update(f1.compute(predictions=predictions, references=labels, average='binary'))
    return result

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [22]:
trainer.train()

C:\Users\Eden\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
500,0.399773


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.39977313232421874, metrics={'train_runtime': 4286.8262, 'train_samples_per_second': 0.933, 'train_steps_per_second': 0.117, 'total_flos': 263111055360000.0, 'train_loss': 0.39977313232421874, 'epoch': 1.0})

In [23]:
trainer.save_model("./best_fake_news_model")
tokenizer.save_pretrained("./best_fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_fake_news_model\\tokenizer_config.json',
 './best_fake_news_model\\tokenizer.json')